In [11]:
import pandas as pd
import datasets
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
import torch

In [12]:
import torch._dynamo
torch._dynamo.config.suppress_errors = True


In [13]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce GTX 1650


In [14]:
start = int(input("Enter Start Year: "))
end = int(input("Enter Ending Year: "))

In [15]:
# =======================
# 🔥 STEP 1 — LOAD + CLEAN FAST
# =======================

def clean_text(t):
    t = str(t)
    t = t.replace("\n", " ").replace("\t", " ")
    t = " ".join(t.split())     # collapse spaces
    return t


df = pd.read_parquet(
    f"D:/LPA_MTech_Project/Enriched_Datasets/SupremeCourt_Combined_{start}_{end}_enriched.parquet"
)


In [16]:

# Keep needed cols
df = df[["text", "verdict_label"]].dropna()

# FAST cleaning BEFORE dataset creation
df["text"] = df["text"].apply(clean_text)

# Hard truncate raw text BEFORE tokenization
df["text"] = df["text"].str[:2500]     # 2500 chars ≈ 512 tokens → FASTEST

df = df.rename(columns={"verdict_label": "label"})
df["label"] = df["label"].astype(int)
df.reset_index(drop=True, inplace=True)

# HF Dataset
dataset = Dataset.from_pandas(df)
dataset = dataset.cast_column("label", datasets.Value("int64"))
dataset = dataset.train_test_split(test_size=0.15, seed=42)

Casting the dataset: 100%|██████████| 4061/4061 [00:00<00:00, 539738.53 examples/s]


In [17]:
# =======================
# 🔥 STEP 2 — MODEL + TOKENIZER (FAST)
# =======================

model_name = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [18]:
# =======================
# 🔥 STEP 3 — TOKENIZE (FAST)
# =======================

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized = dataset.map(tokenize, batched=True, batch_size=128)  # large batch = faster map
tokenized = tokenized.remove_columns(["text"])
tokenized.set_format("torch")


Map: 100%|██████████| 610/610 [00:00<00:00, 1467.78 examples/s]


In [19]:
# =======================
# 🔥 STEP 4 — FAST TRAINING ARGS
# =======================

training_args = TrainingArguments(
    output_dir="legalbert_verdict",
    num_train_epochs=3,

    per_device_train_batch_size=2,         # faster
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=1,         # faster

    learning_rate=2e-5,
    warmup_steps=200,
    weight_decay=0.01,

    fp16=True,
    logging_steps=50,

    eval_strategy="epoch",
    save_strategy="epoch",

    dataloader_num_workers=0,
    gradient_checkpointing=False,          # faster!!
    torch_compile=False, 
)

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
)

In [21]:
trainer.train()


W1117 18:38:33.879787 22244 Lib\site-packages\torch\_dynamo\convert_frame.py:1233] WON'T CONVERT forward d:\LPA_MTech_Project\lpavenv\lib\site-packages\transformers\models\bert\modeling_bert.py line 1639 
W1117 18:38:33.879787 22244 Lib\site-packages\torch\_dynamo\convert_frame.py:1233] due to: 
W1117 18:38:33.879787 22244 Lib\site-packages\torch\_dynamo\convert_frame.py:1233] Traceback (most recent call last):
W1117 18:38:33.879787 22244 Lib\site-packages\torch\_dynamo\convert_frame.py:1233]   File "d:\LPA_MTech_Project\lpavenv\lib\site-packages\torch\_dynamo\convert_frame.py", line 1164, in __call__
W1117 18:38:33.879787 22244 Lib\site-packages\torch\_dynamo\convert_frame.py:1233]     result = self._inner_convert(
W1117 18:38:33.879787 22244 Lib\site-packages\torch\_dynamo\convert_frame.py:1233]   File "d:\LPA_MTech_Project\lpavenv\lib\site-packages\torch\_dynamo\convert_frame.py", line 547, in __call__
W1117 18:38:33.879787 22244 Lib\site-packages\torch\_dynamo\convert_frame.py:1233

{'loss': 0.7164, 'grad_norm': 9.382205963134766, 'learning_rate': 4.7e-06, 'epoch': 0.03}


                                        
  0%|          | 0/5178 [07:47<?, ?it/s]            

{'loss': 0.6963, 'grad_norm': 25.444183349609375, 'learning_rate': 9.7e-06, 'epoch': 0.06}


                                        
  0%|          | 0/5178 [09:03<?, ?it/s]            

...

{'loss': 0.8344, 'grad_norm': 13.35752010345459, 'learning_rate': 1.820409803133789e-05, 'epoch': 0.38}


                                        
  0%|          | 0/5178 [22:40<?, ?it/s]            

{'loss': 0.5988, 'grad_norm': 0.31498152017593384, 'learning_rate': 1.8003214142225796e-05, 'epoch': 0.41}


  0%|          | 0/5178 [23:19<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
print(trainer.evaluate())
